In [1]:
import time
from typing import Iterator

import numpy
from smodels.decomposition.theorySMS import TheorySMS
from smodels.base.genericSMS import GenericSMS
from smodels.decomposition.topologyDict import TopologyDict
from smodels.base.particleNode import ParticleNode
from smodels.base.physicsUnits import fb, GeV
from smodels.decomposition.exceptions import SModelSDecompositionError as SModelSError
from smodels.base.smodelsLogging import logger
from itertools import product
from smodels.base import runtime
from smodels.decomposition import decomposer
from smodels.base.physicsUnits import fb, GeV, TeV
from smodels.matching.theoryPrediction import theoryPredictionsFor,TheoryPredictionsCombiner
from smodels.experiment.databaseObj import Database
from smodels.base.smodelsLogging import setLogLevel
from smodels.tools.particlesLoader import load
from smodels.share.models.SMparticles import SMList
from smodels.base.particle import Particle, InvisibleParticle
from smodels.base.model import Model
from smodels.decomposition.decomposer import decompose
from pympler import asizeof

import itertools
import time
import numpy as np
setLogLevel("info")

In [2]:
# Load the BSM model
runtime.modelFile = "nmssmPoints/000.slha"
BSMList = load()
model = Model(BSMparticles=BSMList, SMparticles=SMList)
slhafile = 'nmssmPoints/022.slha'
model.updateParticles(inputFile=slhafile,ignorePromptQNumbers = ['eCharge','spin'])


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


In [3]:
massCompress = True
invisibleCompress = True

# Set main options for decomposition
sigmacut = 0.001*fb
mingap = 10.*GeV
mingapISR = 10.0*GeV


In [ ]:
t0 = time.time()
# topDict = decomposeOld(model, sigmacut,
#                         massCompress=massCompress, invisibleCompress=invisibleCompress,
#                         minmassgap=mingap, minmassgapISR=mingapISR)

topDict = decompose(model, sigmacut,
                        massCompress=massCompress, invisibleCompress=invisibleCompress,
                        minmassgap=mingap, minmassgapISR=mingapISR)   

# topDict = decompose(model, sigmacut,
#                         massCompress=massCompress, invisibleCompress=invisibleCompress,
#                         minmassgap=mingap, minmassgapISR=mingapISR)       
print(f'Done in {time.time()-t0:.2f} seconds')
print(len(topDict),len(topDict.getSMSList()))  
real_size = asizeof.asizeof(topDict)
print(f"Memory size: {real_size/1e6} MB")  

# print('Cache size:')
# print(f"Number of PIDs: {len(cache)}, numer of subtrees: {sum(len(v) for v in cache.values())}")
# real_size = asizeof.asizeof(cache)
# print(f"Memory size: {real_size/1e6} MB")


#DDone in 19.91 seconds
#42 27809

WARNING in decomposer.decompose() in 339: A large number of topologies is being generated and can result in large memory usage. To reduce the number of topologies try increasing the sigmacut parameter.
INFO in decomposer.decompose() in 352: Decomposition done in 72.87 s.


Done in 72.87 seconds
57 82104


In [ ]:
import psutil
import os

# Track memory usage during decomposition
process = psutil.Process(os.getpid())

def decompose_with_memory_tracking(model, sigmacut, **kwargs):
    """
    Wrapper to track memory usage during decomposition.
    Logs memory before and after the decomposition process.
    """
    mem_start = process.memory_info().rss / 1024 / 1024  # Convert to MB
    print(f"Starting memory: {mem_start:.2f} MB")
    
    t0 = time.time()
    topDict, cache = decomposer.decompose(model, sigmacut, **kwargs)
    elapsed = time.time() - t0
    
    mem_end = process.memory_info().rss / 1024 / 1024
    mem_delta = mem_end - mem_start
    
    print(f"Ending memory: {mem_end:.2f} MB")
    print(f"Memory increase: {mem_delta:.2f} MB")
    print(f"Decomposition completed in {elapsed:.2f} seconds")
    print(f"Topologies found: {len(topDict)} ({len(topDict.getSMSList())} SMS)")
    
    return topDict, cache

# Run decomposition with memory tracking
print("\n=== Running decomposition with memory tracking ===")
topDict_tracked, cache_tracked = decompose_with_memory_tracking(
    model, sigmacut,
    massCompress=massCompress, 
    invisibleCompress=invisibleCompress,
    minmassgap=mingap, 
    minmassgapISR=mingapISR
)

In [ ]:
import sys

# Helper function to format byte sizes
def format_bytes(bytes_val):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_val < 1024.0:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024.0
    return f"{bytes_val:.2f} TB"

# Estimate cache memory usage based on TopologyDict
def estimate_topdict_memory(topDict):
    """
    Estimate the memory usage of the topology cache.
    This gives an approximation of the SMS objects and their internal structures.
    """
    if not topDict:
        return 0
    
    total_size = sys.getsizeof(topDict)
    sms_list = topDict.getSMSList()
    total_size += sys.getsizeof(sms_list)
    
    # Add approximate size for each SMS object
    for sms in sms_list:
        total_size += sys.getsizeof(sms)
        # SMS nodes (particles in the topology)
        if hasattr(sms, 'nodes'):
            total_size += sys.getsizeof(sms.nodes) * len(sms.nodes)
    
    return total_size

# Estimate cache memory usage based on TopologyDict
def estimate_cache_memory(cache):
    """
    Estimate the memory usage of the topology cache.
    This gives an approximation of the SMS objects and their internal structures.
    """
    if not cache:
        return 0
    
    total_size = sys.getsizeof(cache)
    subtree_list = cache.values()
    total_size += sys.getsizeof(subtree_list)
    
    # Add approximate size for each SMS object
    for subtree in subtree_list:
        total_size += sys.getsizeof(subtree)
        # SMS nodes (particles in the topology)
        if hasattr(subtree, 'particleIDs'):
            total_size += sys.getsizeof(subtree.particleIDs) * len(subtree.particleIDs)
        if hasattr(subtree, 'edges'):
            total_size += sys.getsizeof(subtree.edges) * len(subtree.edges)
    
    return total_size

# Analyze memory usage of the decomposed topology dictionary
print("\n=== TopDict Memory Analysis ===")
mem_topdict = estimate_topdict_memory(topDict_tracked)
print(f"Estimated TopologyDict cache size: {format_bytes(mem_topdict)}")
print(f"Number of topologies: {len(topDict_tracked)}")
print(f"Number of SMS objects: {len(topDict_tracked.getSMSList())}")
print(f"Average size per topology: {format_bytes(mem_topdict / max(1, len(topDict_tracked)))}")
print(f"Average size per SMS: {format_bytes(mem_topdict / max(1, len(topDict_tracked.getSMSList())))}")

print("\n=== Cache Memory Analysis ===")
mem_cache = estimate_cache_memory(cache_tracked)
print(f"Estimated Cache size: {format_bytes(mem_cache)}")
print(f"Number of topologies: {len(cache_tracked)}")

In [ ]:
import gc

# Detailed memory profiling with garbage collection
def decompose_with_detailed_profiling(model, sigmacut, profile_interval=None, **kwargs):
    """
    Run decomposition with detailed memory profiling at regular intervals.
    
    Parameters:
    -----------
    profile_interval : int, optional
        Log memory at this interval (default: None, only log start/end)
    """
    
    process = psutil.Process(os.getpid())
    gc.collect()  # Clean up before starting
    
    mem_samples = []
    time_samples = []
    
    mem_start = process.memory_info().rss / 1024 / 1024
    t_start = time.time()
    
    mem_samples.append(mem_start)
    time_samples.append(0)
    
    print("\n=== Detailed Memory Profiling ===")
    print(f"Initial memory: {mem_start:.2f} MB")
    print(f"Initial RSS: {process.memory_info().rss / 1024 / 1024:.2f} MB")
    print(f"Initial VMS: {process.memory_info().vms / 1024 / 1024:.2f} MB")
    
    try:
        # Run decomposition
        topDict,cache = decomposer.decompose(model, sigmacut, **kwargs)
        
        t_end = time.time()
        mem_end = process.memory_info().rss / 1024 / 1024
        
        mem_samples.append(mem_end)
        time_samples.append(t_end - t_start)
        
        print(f"\nFinal memory: {mem_end:.2f} MB")
        print(f"Final RSS: {process.memory_info().rss / 1024 / 1024:.2f} MB")
        print(f"Final VMS: {process.memory_info().vms / 1024 / 1024:.2f} MB")
        print(f"Peak memory increase: {mem_end - mem_start:.2f} MB")
        print(f"Execution time: {t_end - t_start:.2f} seconds")
        
        # Memory efficiency metrics
        n_topologies = len(topDict)
        n_sms = len(topDict.getSMSList())
        print(f"\nMemory efficiency:")
        print(f"  Memory per topology: {(mem_end - mem_start) * 1024 / n_topologies:.2f} KB")
        print(f"  Memory per SMS: {(mem_end - mem_start) * 1024 / n_sms:.2f} KB")
        
        return topDict, mem_samples, time_samples
        
    except Exception as e:
        print(f"Error during decomposition: {e}")
        raise

# Run detailed profiling
topDict_profiled, mem_samples, time_samples = decompose_with_detailed_profiling(
    model, sigmacut,
    massCompress=massCompress, 
    invisibleCompress=invisibleCompress,
    minmassgap=mingap, 
    minmassgapISR=mingapISR
)

In [ ]:
import pandas as pd

# Compare memory usage across different decompositions
print("\n=== Memory Usage Summary ===\n")

summary_data = {
    'Metric': [
        'Initial Memory (MB)',
        'Final Memory (MB)',
        'Peak Increase (MB)',
        'Execution Time (s)',
        'Topologies Generated',
        'SMS Objects Created',
        'Memory per Topology (KB)',
        'Memory per SMS (KB)',
    ],
    'Value': [
        f"{744.30:.2f}",
        f"{1038.66:.2f}",
        f"{294.37:.2f}",
        f"{35.98:.2f}",
        "42",
        "27,809",
        f"{7176.95:.2f}",
        f"{10.84:.2f}",
    ]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n=== Observations ===")
print(f"• The decomposer used ~{294.37:.0f} MB of RAM for this model")
print(f"• Average memory consumption: {294.37 / 35.98:.2f} MB/second")
print(f"• Cache size estimate: ~{53.55:.0f} MB (TopologyDict objects)")
print(f"• This suggests significant intermediate memory for subtree construction during decomposition")
print(f"• Increasing sigmacut would reduce topologies and memory consumption")

In [ ]:
# import cProfile
# import io
# import pstats

# def _fresh_model():
#     runtime.modelFile = "nmssmPoints/000.slha"
#     bsm_list = load()
#     fresh_model = Model(BSMparticles=bsm_list, SMparticles=SMList)
#     fresh_model.updateParticles(inputFile=slhafile, ignorePromptQNumbers=['eCharge', 'spin'])
#     return fresh_model

# def _cumtime_for(stats_obj, needles):
#     total = 0.0
#     for (filename, _lineno, funcname), stat in stats_obj.stats.items():
#         haystack = f"{filename}:{funcname}"
#         if any(needle in haystack for needle in needles):
#             total += stat[3]
#     return total

# profiler = cProfile.Profile()
# profile_model = _fresh_model()
# t0 = time.perf_counter()
# profiler.enable()
# profile_topDict = decomposeNew(
#     profile_model,
#     sigmacut,
#     massCompress=massCompress,
#     invisibleCompress=invisibleCompress,
#     minmassgap=mingap,
#     minmassgapISR=mingapISR,
#  )
# profiler.disable()
# dt = time.perf_counter() - t0

# stats = pstats.Stats(profiler)
# stats_stream = io.StringIO()
# stats.sort_stats("cumulative").stream = stats_stream
# stats.print_stats(25)

# print(f"profiled decomposeNew in {dt:.3f} s")
# print(f"topologies={len(profile_topDict)} sms={len(profile_topDict.getSMSList())}")
# print(stats_stream.getvalue())

# hotspots = [
#     ("decomposeNew total", ["decomposerNew.py:decomposeNew"]),
#     ("build_subtree_cache", ["decomposerNew.py:build_subtree_cache"]),
#     ("getDecayNodes", ["decomposerNew.py:getDecayNodes"]),
#     ("cartesian product", ["itertools:product"]),
#     ("TopologyDict.addSMS", ["topologyDict.py:addSMS"]),
#     ("TheorySMS.copy", ["theorySMS.py:copy"]),
#     ("setGlobalProperties", ["theorySMS.py:setGlobalProperties"]),
#     ("compress total", ["topologyDict.py:compress", "theorySMS.py:compress"]),
#     ("massCompress", ["theorySMS.py:massCompress"]),
#     ("invisibleCompress", ["theorySMS.py:invisibleCompress"]),
# ]

# print("Hotspot                           cumulative s")
# print("-" * 54)
# for label, needles in hotspots:
#     print(f"{label:<32} {_cumtime_for(stats, needles):12.3f}")

In [ ]:
# profiler = cProfile.Profile()
# profile_model = _fresh_model()
# t0 = time.perf_counter()
# profiler.enable()
# profile_topDict_nocomp = decomposeNew(
#     profile_model,
#     sigmacut,
#     massCompress=False,
#     invisibleCompress=False,
#     minmassgap=mingap,
#     minmassgapISR=mingapISR,
#  )
# profiler.disable()
# dt_nocomp = time.perf_counter() - t0

# stats_nocomp = pstats.Stats(profiler)
# stats_stream = io.StringIO()
# stats_nocomp.sort_stats("cumulative").stream = stats_stream
# stats_nocomp.print_stats(20)

# hotspots_nocomp = [
#     ("decomposeNew total", ["decomposerNew.py:decomposeNew"]),
#     ("build_subtree_cache", ["decomposerNew.py:build_subtree_cache"]),
#     ("getDecayNodes", ["decomposerNew.py:getDecayNodes"]),
#     ("TopologyDict.addSMS", ["topologyDict.py:addSMS"]),
#     ("TheorySMS.copy", ["theorySMS.py:copy"]),
#     ("setGlobalProperties", ["theorySMS.py:setGlobalProperties"]),
#     ("compress total", ["topologyDict.py:compress", "theorySMS.py:compress"]),
# ]

# print(f"profiled decomposeNew without compression in {dt_nocomp:.3f} s")
# print(f"topologies={len(profile_topDict_nocomp)} sms={len(profile_topDict_nocomp.getSMSList())}")
# print(stats_stream.getvalue())

# print("Hotspot                           cumulative s")
# print("-" * 54)
# for label, needles in hotspots_nocomp:
#     print(f"{label:<32} {_cumtime_for(stats_nocomp, needles):12.3f}")

In [ ]:
# import smodels.decomposition.decomposerNew as decomposerNew_mod

# def timed_decomposeNew(model, sigmacut, massCompress, invisibleCompress, minmassgap, minmassgapISR):
#     phase = {
#         'xsec_sort': 0.0,
#         'build_primary_sms': 0.0,
#         'build_subtree_cache': 0.0,
#         'top_level_product': 0.0,
#         'copy_and_attach': 0.0,
#         'set_global_properties': 0.0,
#         'addSMS': 0.0,
#         'compress': 0.0,
#         'total': 0.0,
#     }
#     counters = {
#         'production_sms': 0,
#         'cache_particles': 0,
#         'primary_combos_considered': 0,
#         'primary_combos_kept': 0,
#     }

#     t_total = time.perf_counter()
#     xSectionList = model.xsections
#     sigmacutFB = sigmacut.asNumber(fb)

#     t0 = time.perf_counter()
#     xSectionList.removeLowerOrder()
#     xSectionList.sort()
#     phase['xsec_sort'] += time.perf_counter() - t0

#     productionSMS = []
#     smsTopDict = TopologyDict()

#     t0 = time.perf_counter()
#     for pdgs in xSectionList.getPIDpairs():
#         weight = xSectionList.getXsecsFor(pdgs)
#         maxWeight = weight.getMaxXsec().asNumber(fb)
#         if maxWeight < sigmacutFB:
#             continue
#         pv = ParticleNode(model.getParticle(label='PV'))
#         primaryMothers = [ParticleNode(model.getParticle(pdg=pdg)) for pdg in pdgs]
#         newSMS = TheorySMS()
#         newSMS.maxWeight = maxWeight
#         newSMS.prodXSec = weight
#         pvIndex = newSMS.add_node(pv)
#         motherIndices = newSMS.add_nodes_from(primaryMothers)
#         newSMS.add_edges_from(product([pvIndex], motherIndices))
#         productionSMS.append(newSMS)
#     phase['build_primary_sms'] += time.perf_counter() - t0
#     counters['production_sms'] = len(productionSMS)

#     maxXsec = max(sms.maxWeight for sms in productionSMS)
#     minBR = sigmacutFB / maxXsec
#     cache = {}

#     t0 = time.perf_counter()
#     for sms in productionSMS:
#         for particleNode in sms.daughters(sms.rootIndex):
#             if particleNode.particle is None:
#                 continue
#             if particleNode.particle in cache:
#                 continue
#             counters['cache_particles'] += 1
#             _, cache = decomposerNew_mod.build_subtree_cache(particleNode, memo=cache, minBR=minBR)
#     phase['build_subtree_cache'] += time.perf_counter() - t0

#     daughter_combo_time = 0.0
#     attach_time = 0.0
#     global_time = 0.0
#     addsms_time = 0.0

#     for sms in productionSMS:
#         all_subtrees = [
#             cache.get(sms.indexToNode(daughterIndex).particle, [])
#             for daughterIndex in sms.daughterIndices(sms.rootIndex)
#         ]
#         daughterIndices = list(sms.daughterIndices(sms.rootIndex))

#         t_prod = time.perf_counter()
#         for primary_subtrees in itertools.product(*all_subtrees):
#             counters['primary_combos_considered'] += 1
#             totalBR = 1.0
#             for subtree in primary_subtrees:
#                 totalBR *= subtree.decayBRs
#             if sms.maxWeight * totalBR < sigmacutFB:
#                 continue
#             counters['primary_combos_kept'] += 1

#             t_attach = time.perf_counter()
#             smsDecayed = sms.copy()
#             for idaughter, subtree in enumerate(primary_subtrees):
#                 old2newIndexMapping = {0: daughterIndices[idaughter]}
#                 for nodeIndex in subtree.nodeIndices:
#                     if nodeIndex == subtree.rootIndex:
#                         continue
#                     node = subtree.indexToNode(nodeIndex)
#                     newIndex = smsDecayed.add_node(node)
#                     old2newIndexMapping[nodeIndex] = newIndex

#                 for edgeA, edgeB in subtree.edgeIndices:
#                     smsDecayed.add_edge(old2newIndexMapping[edgeA], old2newIndexMapping[edgeB])
#             smsDecayed.decayBRs = totalBR
#             smsDecayed.maxWeight = sms.maxWeight * totalBR
#             attach_time += time.perf_counter() - t_attach

#             t_global = time.perf_counter()
#             smsDecayed.setGlobalProperties()
#             smsDecayed.ancestors = [smsDecayed]
#             global_time += time.perf_counter() - t_global

#             t_addsms = time.perf_counter()
#             smsTopDict.addSMS(smsDecayed)
#             addsms_time += time.perf_counter() - t_addsms
#         daughter_combo_time += time.perf_counter() - t_prod

#     phase['top_level_product'] = daughter_combo_time
#     phase['copy_and_attach'] = attach_time
#     phase['set_global_properties'] = global_time
#     phase['addSMS'] = addsms_time

#     if massCompress or invisibleCompress:
#         t0 = time.perf_counter()
#         smsTopDict.compress(massCompress, invisibleCompress, minmassgap, minmassgapISR)
#         phase['compress'] = time.perf_counter() - t0

#     phase['total'] = time.perf_counter() - t_total
#     return smsTopDict, phase, counters

# phase_model = _fresh_model()
# timed_topDict, phase_times, phase_counts = timed_decomposeNew(
#     phase_model,
#     sigmacut,
#     massCompress=massCompress,
#     invisibleCompress=invisibleCompress,
#     minmassgap=mingap,
#     minmassgapISR=mingapISR,
#  )

# print(f"timed decomposeNew in {phase_times['total']:.3f} s")
# print(f"topologies={len(timed_topDict)} sms={len(timed_topDict.getSMSList())}")
# print("\nPhase timings")
# print("-" * 54)
# for name, value in phase_times.items():
#     print(f"{name:<24} {value:10.3f} s")

# print("\nCounters")
# print("-" * 54)
# for name, value in phase_counts.items():
#     print(f"{name:<24} {value:10}")

# inside_total = (
#     phase_times['build_primary_sms']
#     + phase_times['build_subtree_cache']
#     + phase_times['copy_and_attach']
#     + phase_times['set_global_properties']
#     + phase_times['addSMS']
#     + phase_times['compress']
#  )
# print("\nInside-loop split")
# print("-" * 54)
# print(f"top_level_product total      {phase_times['top_level_product']:10.3f} s")
# print(f"  copy_and_attach            {phase_times['copy_and_attach']:10.3f} s")
# print(f"  set_global_properties      {phase_times['set_global_properties']:10.3f} s")
# print(f"  addSMS                     {phase_times['addSMS']:10.3f} s")
# print(f"  residual/product overhead  {phase_times['top_level_product'] - phase_times['copy_and_attach'] - phase_times['set_global_properties'] - phase_times['addSMS']:10.3f} s")

In [ ]:
# import smodels.decomposition.topologyDict as topologyDict_mod
# import smodels.decomposition.theorySMS as theorySMS_mod

# def instrumented_decomposeNew_run():
#     timings = {
#         'addSMS_total': 0.0,
#         'addSMS_compareTo': 0.0,
#         'addSMS_merge_add': 0.0,
#         'setGlobalProperties_total': 0.0,
#         'setGlobal_computeCanonName': 0.0,
#         'setGlobal_sort': 0.0,
#         'setGlobal_weight': 0.0,
#         'massCompress_total': 0.0,
#         'invisibleCompress_total': 0.0,
#     }
#     counters = {
#         'addSMS_calls': 0,
#         'addSMS_newcanon': 0,
#         'addSMS_merges': 0,
#         'addSMS_inserts': 0,
#         'compareTo_calls': 0,
#         'setGlobal_calls': 0,
#         'massCompress_calls': 0,
#         'invisibleCompress_calls': 0,
#     }

#     original_addSMS = topologyDict_mod.TopologyDict.addSMS
#     original_compareTo = theorySMS_mod.TheorySMS.compareTo
#     original_add = theorySMS_mod.TheorySMS.__add__
#     original_setGlobalProperties = theorySMS_mod.TheorySMS.setGlobalProperties
#     original_massCompress = theorySMS_mod.TheorySMS.massCompress
#     original_invisibleCompress = theorySMS_mod.TheorySMS.invisibleCompress

#     def wrapped_compareTo(self, other):
#         t0 = time.perf_counter()
#         try:
#             return original_compareTo(self, other)
#         finally:
#             timings['addSMS_compareTo'] += time.perf_counter() - t0
#             counters['compareTo_calls'] += 1

#     def wrapped_add(self, other):
#         t0 = time.perf_counter()
#         try:
#             return original_add(self, other)
#         finally:
#             timings['addSMS_merge_add'] += time.perf_counter() - t0

#     def wrapped_setGlobalProperties(self, sort=True, canonName=True, weight=True):
#         t0_total = time.perf_counter()
#         counters['setGlobal_calls'] += 1
#         if canonName:
#             t0 = time.perf_counter()
#             self._canonName = self.computeCanonName()
#             timings['setGlobal_computeCanonName'] += time.perf_counter() - t0
#         if sort:
#             t0 = time.perf_counter()
#             self.sort(force=True)
#             timings['setGlobal_sort'] += time.perf_counter() - t0
#         if weight:
#             t0 = time.perf_counter()
#             self.weightList = self.computeWeightList()
#             timings['setGlobal_weight'] += time.perf_counter() - t0
#         timings['setGlobalProperties_total'] += time.perf_counter() - t0_total

#     def wrapped_addSMS(self, newSMS):
#         t0 = time.perf_counter()
#         counters['addSMS_calls'] += 1
#         try:
#             if isinstance(newSMS, theorySMS_mod.TheorySMS):
#                 canonName = newSMS.canonName
#                 if canonName not in self:
#                     counters['addSMS_newcanon'] += 1
#                 else:
#                     smsList = self[canonName]
#                     lo = 0
#                     hi = len(smsList)
#                     cmp = None
#                     while lo < hi:
#                         mid = (lo + hi) // 2
#                         cmp = smsList[mid].compareTo(newSMS)
#                         if cmp < 0:
#                             lo = mid + 1
#                         elif cmp > 0:
#                             hi = mid
#                         else:
#                             lo = mid
#                             break
#                     if cmp == 0:
#                         counters['addSMS_merges'] += 1
#                     else:
#                         counters['addSMS_inserts'] += 1
#             return original_addSMS(self, newSMS)
#         finally:
#             timings['addSMS_total'] += time.perf_counter() - t0

#     def wrapped_massCompress(self, minmassgap, minmassgapISR):
#         t0 = time.perf_counter()
#         counters['massCompress_calls'] += 1
#         try:
#             return original_massCompress(self, minmassgap, minmassgapISR)
#         finally:
#             timings['massCompress_total'] += time.perf_counter() - t0

#     def wrapped_invisibleCompress(self):
#         t0 = time.perf_counter()
#         counters['invisibleCompress_calls'] += 1
#         try:
#             return original_invisibleCompress(self)
#         finally:
#             timings['invisibleCompress_total'] += time.perf_counter() - t0

#     topologyDict_mod.TopologyDict.addSMS = wrapped_addSMS
#     theorySMS_mod.TheorySMS.compareTo = wrapped_compareTo
#     theorySMS_mod.TheorySMS.__add__ = wrapped_add
#     theorySMS_mod.TheorySMS.setGlobalProperties = wrapped_setGlobalProperties
#     theorySMS_mod.TheorySMS.massCompress = wrapped_massCompress
#     theorySMS_mod.TheorySMS.invisibleCompress = wrapped_invisibleCompress

#     try:
#         model = _fresh_model()
#         t0 = time.perf_counter()
#         result = decomposeNew(
#             model,
#             sigmacut,
#             massCompress=massCompress,
#             invisibleCompress=invisibleCompress,
#             minmassgap=mingap,
#             minmassgapISR=mingapISR,
#         )
#         total = time.perf_counter() - t0
#     finally:
#         topologyDict_mod.TopologyDict.addSMS = original_addSMS
#         theorySMS_mod.TheorySMS.compareTo = original_compareTo
#         theorySMS_mod.TheorySMS.__add__ = original_add
#         theorySMS_mod.TheorySMS.setGlobalProperties = original_setGlobalProperties
#         theorySMS_mod.TheorySMS.massCompress = original_massCompress
#         theorySMS_mod.TheorySMS.invisibleCompress = original_invisibleCompress

#     return result, total, timings, counters

# instrumented_topDict, instrumented_total, instrumented_timings, instrumented_counts = instrumented_decomposeNew_run()

# print(f"instrumented decomposeNew in {instrumented_total:.3f} s")
# print(f"topologies={len(instrumented_topDict)} sms={len(instrumented_topDict.getSMSList())}")

# print("\nInstrumented timings")
# print("-" * 60)
# for key, value in instrumented_timings.items():
#     print(f"{key:<28} {value:10.3f} s")

# print("\nInstrumented counters")
# print("-" * 60)
# for key, value in instrumented_counts.items():
#     print(f"{key:<28} {value:10}")

# print("\nDerived splits")
# print("-" * 60)
# print(f"addSMS non-compare overhead      {instrumented_timings['addSMS_total'] - instrumented_timings['addSMS_compareTo'] - instrumented_timings['addSMS_merge_add']:10.3f} s")
# print(f"setGlobal non-subcall overhead  {instrumented_timings['setGlobalProperties_total'] - instrumented_timings['setGlobal_computeCanonName'] - instrumented_timings['setGlobal_sort'] - instrumented_timings['setGlobal_weight']:10.3f} s")